# MAZEWARD VERSUS — Colab 学習サーバー

ローカルの GUI から Colab の GPU 学習を **開始 / 停止 / 監視** するための一式です。

## 手順（上から順に実行）

| | セル | 何をするか |
|---|---|---|
| (1) | Setup | Drive をマウントしてコードを取り込む。**API トークンを Drive に固定** |
| (2) | 制御サーバー起動 | 学習を操作する Flask を立て、応答するまで待つ |
| (3) | ngrok 公開 | 外から届く URL を作る。**GUI に貼る値がここに出ます** |
| (3.5) | ヘルパー | 通信用の関数を定義。**(4)(5) より先に実行** |
| (4) | 死活監視 | 切れたら自動で張り直す。**回したままにしてください** |
| (5) | 状況 / 停止 | 学習の様子を見る・止める |

## API トークンは固定されます

以前は `API.txt` を `/content` に作っていたため、ランタイムが変わるたびに
新しい値になっていました。いまは **Drive の `API.txt` を正本**にしているので、
一度 GUI に貼れば次回以降は同じ値です。

**初回だけ値が変わります。** Drive にまだ `API.txt` が無いためです。
(1) が表示した値を GUI に貼り直してください。以降は変わりません。

## 既に Drive へ送信済みの方へ

以前のバージョンは **すべてのファイルを Drive フォルダの直下に平置き**していました。
そのため Colab 側に `ai/` が無く、(2) が `No such file or directory` で失敗します。
GUI を最新にしたうえで **「コードを Colab へ送信」をもう一度**実行してください。
Drive 直下に残った古い `.py` は削除して構いません。

## つながらないときの見分け方

| 症状 | 意味 | 対処 |
|---|---|---|
| `ERR_NGROK_3200` / `SSL: UNEXPECTED_EOF` | トンネルが落ちた | (4) が自動で張り直します。止まっていたら (3) を再実行 |
| `401` / 認証に失敗 | トークンが違う | (1) の出力を GUI に貼り直す |
| `JSON が返ってきません` | URL が別のものを指している | (3) の URL を貼り直す |
| `No such file or directory` | Drive の構造が古い | GUI から「コードを Colab へ送信」 |


## (1) Setup — コード取り込みと API トークンの固定


In [ ]:
import os, shutil, secrets, pathlib
from google.colab import drive

DRIVE_FOLDER = 'mazeward_colab_rl_ai'      # Drive 側のフォルダ名
COLAB_DIR    = '/content/mazeward_colab_rl_ai'
PORT         = 5558

drive.mount('/content/drive', force_remount=True)
DRIVE_ROOT = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
assert os.path.exists(DRIVE_ROOT), (
    f'Drive に {DRIVE_ROOT} がありません。'
    'ローカル GUI の「Colab・Drive連携 → コードを Colab へ送信」を先に実行してください')

# 毎回まっさらにしてから配置する（古いコードが残ると原因が分からなくなる）
if os.path.exists(COLAB_DIR):
    shutil.rmtree(COLAB_DIR)
shutil.copytree(DRIVE_ROOT, COLAB_DIR,
                ignore=shutil.ignore_patterns('models', 'replays', '__pycache__',
                                              '*.ipynb', 'API.txt', '*.pt'))

# ---- API トークンは Drive に置いて固定する ----
# 以前は /content に作っていたので、ランタイムが変わるたび新しい値になり、
# そのたび GUI へ貼り直す必要があった。Drive に置けば次回もそのまま使える。
drive_token = pathlib.Path(DRIVE_ROOT) / 'API.txt'
if not drive_token.exists():
    drive_token.write_text(secrets.token_hex(16), encoding='utf-8')
    print('API トークンを新規作成しました（次回以降は使い回します）')
API_TOKEN = drive_token.read_text(encoding='utf-8').strip()

local_token = pathlib.Path(COLAB_DIR) / 'ai' / 'API.txt'
local_token.parent.mkdir(parents=True, exist_ok=True)
local_token.write_text(API_TOKEN, encoding='utf-8')

print('コード配置:', COLAB_DIR)
print('API トークン:', API_TOKEN)
print('  ^ GUI の「API トークン」欄に貼ります（Drive 保存済みなので次回も同じ）')

# ---- 取り込めたかを必ず確認する ----
# ここを省くと『No such file or directory』だけが出て、
# Drive の中身がどうなっているのか分からないまま詰まる。
need = pathlib.Path(COLAB_DIR) / 'ai' / 'mazeward_colab_control.py'
if not need.exists():
    print('!! 制御サーバーのファイルがありません:', need)
    print()
    print('Drive フォルダ直下にあるもの（先頭20件）:')
    for q in sorted(pathlib.Path(DRIVE_ROOT).iterdir())[:20]:
        print('   ', q.name + ('/' if q.is_dir() else ''))
    print()
    flat = list(pathlib.Path(DRIVE_ROOT).glob('*.py'))
    if flat:
        print('→ .py が Drive の直下に平置きされています。')
        print('  ローカル GUI を最新にしてから「コードを Colab へ送信」を',
              'もう一度実行してください（ai/ 付きで送るよう修正済みです）。')
        print('  古い平置きファイルは Drive から消して構いません。')
    else:
        print('→ まず GUI の「コードを Colab へ送信」を実行してください。')
    raise SystemExit('コードの取り込みに失敗しています')

n_env = len(list((pathlib.Path(COLAB_DIR) / 'ai' / 'mazeward_env').glob('*.py')))
print(f'確認 OK: 制御サーバーあり / mazeward_env に {n_env} ファイル')


## (2) 制御サーバーを起動 — 生きているか必ず確認する


In [ ]:
import subprocess, sys, time, json, pathlib
import urllib.request, urllib.error

LOG_PATH = '/content/mazeward_control.log'

# 既に動いていたら止める（二重起動するとポートを奪い合う）
subprocess.run(['pkill', '-f', 'mazeward_colab_control.py'], check=False)
time.sleep(1)

log = open(LOG_PATH, 'w', encoding='utf-8')
proc = subprocess.Popen(
    [sys.executable, f'{COLAB_DIR}/ai/mazeward_colab_control.py', '--port', str(PORT)],
    cwd=COLAB_DIR, stdout=log, stderr=subprocess.STDOUT)

# 立ち上がるまで待ち、**実際に応答するか**を確かめる。
# ここを省くと、死んでいるサーバーに ngrok を張って 404 で悩むことになる。
# 制御サーバーは API.txt があると **全ルートに** X-API-Token を要求する。
# ヘッダを付けずに /healthz を叩くと 401 になり、起動しているのに
# 「起動に失敗」と誤判定する（実際にこれで詰まった）。
def _health():
    req = urllib.request.Request(f'http://127.0.0.1:{PORT}/healthz',
                                 headers={'X-API-Token': API_TOKEN})
    with urllib.request.urlopen(req, timeout=3) as r:
        return json.loads(r.read().decode()).get('ok', False)

ok, last_err = False, ''
for _ in range(30):
    time.sleep(1)
    try:
        ok = _health()
        if ok:
            break
    except urllib.error.HTTPError as e:
        last_err = f'HTTP {e.code}'
        if e.code == 401:
            # ここに来るのは API.txt と手元の API_TOKEN がずれているとき
            break
    except Exception as e:
        last_err = str(e)
        if proc.poll() is not None:
            break

if ok:
    print(f'制御サーバー起動 OK  pid={proc.pid}  port={PORT}')
else:
    if last_err == 'HTTP 401':
        print('サーバーは動いていますが、トークンが一致しません。')
        served = pathlib.Path(COLAB_DIR) / 'ai' / 'API.txt'
        print('  サーバーが読む値:', served.read_text(encoding="utf-8").strip()
              if served.exists() else '(API.txt なし)')
        print('  こちらが送った値:', API_TOKEN)
        print('  → (1) をもう一度実行してから、このセルを実行し直してください。')
    else:
        print('起動に失敗しました:', last_err)
        print('ログの末尾:')
        print(open(LOG_PATH, encoding='utf-8').read()[-2000:])
    raise SystemExit('制御サーバーが起動していません')


## (3) ngrok で公開 — ここに出る 2 つを GUI に貼ります

`NGROK_AUTHTOKEN` は次の順に探します。**一度 Drive に置けば次回から入力不要**です。

1. Colab のシークレット（左の鍵アイコン → `NGROK_AUTHTOKEN`）
2. Drive の `ngrok_authtoken.txt`
3. その場で入力（入力したら Drive に保存します）

固定ドメインを持っているなら Drive に `ngrok_domain.txt` を置くと **毎回同じ URL** になります。


In [ ]:
import os, pathlib, getpass, subprocess, sys, json, urllib.request
try:
    from pyngrok import ngrok
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'], check=True)
    from pyngrok import ngrok

DRIVE_ROOT  = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
auth_file   = pathlib.Path(DRIVE_ROOT) / 'ngrok_authtoken.txt'
domain_file = pathlib.Path(DRIVE_ROOT) / 'ngrok_domain.txt'

auth = os.environ.get('NGROK_AUTHTOKEN', '').strip()
if not auth:
    try:
        from google.colab import userdata
        auth = (userdata.get('NGROK_AUTHTOKEN') or '').strip()
    except Exception:
        auth = ''
if not auth and auth_file.exists():
    auth = auth_file.read_text(encoding='utf-8').strip()
if not auth:
    auth = getpass.getpass('ngrok authtoken を入力（次回のため Drive へ保存します）: ').strip()
    if auth:
        auth_file.write_text(auth, encoding='utf-8')
        print('Drive に保存しました:', auth_file)
assert auth, 'ngrok authtoken がありません'
ngrok.set_auth_token(auth)

domain = os.environ.get('NGROK_DOMAIN', '').strip()
if not domain and domain_file.exists():
    domain = domain_file.read_text(encoding='utf-8').strip()

ngrok.kill()                     # 古いトンネルが残ると URL が増えて混乱する
tunnel = ngrok.connect(PORT, domain=domain) if domain else ngrok.connect(PORT)
PUBLIC_URL = tunnel.public_url.replace('http://', 'https://')

# 公開 URL 越しに疎通確認。ngrok の警告ページを避けるヘッダが必須
req = urllib.request.Request(PUBLIC_URL + '/healthz', headers={
    'ngrok-skip-browser-warning': 'true',
    'X-API-Token': API_TOKEN,
})
try:
    with urllib.request.urlopen(req, timeout=15) as r:
        reachable = json.loads(r.read().decode()).get('ok', False)
except Exception as e:
    reachable = False
    print('公開 URL への疎通に失敗:', e)

print('=' * 68)
print('  GUI（Colab・Drive連携 タブ）に貼る値')
print('=' * 68)
print('  ngrok URL   :', PUBLIC_URL)
print('  API トークン:', API_TOKEN)
print('=' * 68)
print('  疎通:', 'OK' if reachable else 'NG（(2) のログを確認してください）')
if not domain:
    print()
    print('  ヒント: 固定ドメインがあるなら Drive に ngrok_domain.txt を作って')
    print('        中にドメイン名を書いておくと、URL が毎回同じになります。')


## (3.5) ヘルパー — ここで通信用の関数を定義します

**このセルは (4)(5) より先に実行してください。**
以前は `_get` を死活監視のセル（無限ループ）の中で定義していたため、
(5) を単独で実行すると `NameError: name '_get' is not defined` になりました。
終了するセルで定義しておけば、どのセルからでも使えます。


In [ ]:
import json, time, urllib.request, urllib.error

def _get(path, timeout=10):
    req = urllib.request.Request(f'http://127.0.0.1:{PORT}' + path,
                                 headers={'X-API-Token': API_TOKEN})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())

def _post(path, body=None, timeout=30):
    data = json.dumps(body or {}).encode()
    req = urllib.request.Request(f'http://127.0.0.1:{PORT}' + path, data=data,
                                 method='POST',
                                 headers={'Content-Type': 'application/json',
                                          'X-API-Token': API_TOKEN})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())

def tunnel_alive(url, timeout=10):
    """公開 URL 越しに生きているか。ngrok が落ちると False。"""
    req = urllib.request.Request(url + '/healthz', headers={
        'ngrok-skip-browser-warning': 'true', 'X-API-Token': API_TOKEN})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return json.loads(r.read().decode()).get('ok', False)
    except Exception:
        return False

def reconnect_tunnel():
    """ngrok を張り直して新しい公開 URL を返す。"""
    from pyngrok import ngrok
    try:
        ngrok.kill()
    except Exception:
        pass
    time.sleep(2)
    t = ngrok.connect(PORT, domain=domain) if domain else ngrok.connect(PORT)
    return t.public_url.replace('http://', 'https://')

print('ヘルパーを定義しました: _get / _post / tunnel_alive / reconnect_tunnel')
print('制御サーバー:', _get('/healthz'))


## (4) 死活監視 — **このセルは回したままにしてください**

Colab のランタイムが揺れると ngrok が落ちます。GUI 側には
`ERR_NGROK_3200`（endpoint is offline）や `SSL: UNEXPECTED_EOF` として見えます。

このセルは 30 秒ごとに **公開 URL 越し** で生存を確認し、落ちていたら
**自動で張り直します**。固定ドメインを使っていれば URL は変わらないので、
GUI 側は何もしなくてよく、そのまま学習が続きます。
（固定ドメインでない場合は新しい URL を表示するので、GUI に貼り直してください）


In [ ]:
import datetime

print('死活監視を開始します（止めるにはこのセルを中断）')
print('監視対象:', PUBLIC_URL)
fails = 0
while True:
    now = datetime.datetime.now().strftime('%H:%M:%S')
    # ① 制御サーバー自体（ローカル）
    try:
        st = _get('/colab/status')
        live = st.get('live') or {}
        latest = st.get('latest') or {}
        gen = live.get('episode') if live.get('episode') is not None else latest.get('episode', '-')
        step = f"{live.get('step')}/{live.get('max_steps')}" if live.get('step') else '-'
        local_ok = True
    except Exception as e:
        local_ok, gen, step = False, '-', '-'
        print(f'[{now}] 制御サーバーに繋がりません: {e}', flush=True)

    # ② 外から届くか（ngrok）
    if local_ok:
        if tunnel_alive(PUBLIC_URL):
            fails = 0
            print(f'[{now}] OK 学習中={st.get("is_running")} 世代={gen} '
                  f'ステップ={step} 記録={st.get("gen_count")}世代', flush=True)
        else:
            fails += 1
            print(f'[{now}] トンネルが落ちています（{fails} 回目）。張り直します…', flush=True)
            try:
                new_url = reconnect_tunnel()
                if new_url != PUBLIC_URL:
                    PUBLIC_URL = new_url
                    print(f'[{now}] !! 新しい URL: {PUBLIC_URL}', flush=True)
                    print('     GUI の ngrok URL を貼り替えてください', flush=True)
                else:
                    print(f'[{now}] 復旧しました（URL は同じ）', flush=True)
                fails = 0
            except Exception as e:
                print(f'[{now}] 張り直しに失敗: {e}', flush=True)
    time.sleep(30)


## (5) 状況の確認 / 学習の停止（必要なときだけ）


In [ ]:
# --- いまの状況（(3.5) を先に実行しておくこと） ---
st = _get('/colab/status')
print(json.dumps({k: v for k, v in st.items() if k != 'logs'},
                 ensure_ascii=False, indent=2))
print()
print('--- 直近のログ ---')
for line in (st.get('logs') or [])[-15:]:
    print(f"{line.get('time')} [{line.get('tag')}] {line.get('text')}")
print()
print('公開 URL 越しの疎通:', 'OK' if tunnel_alive(PUBLIC_URL) else 'NG（(4) が張り直します）')


In [ ]:
# --- 学習を止める（GUI の「停止」と同じ） ---
print(_post('/colab/stop'))


## 学習ログはどこに残るか

`trainer_pb.py` が書き込みのたびに Drive の `mazeward_checkpoints/` へコピーします。
**ランタイムが切れてもグラフの元データは消えません。**
ローカル GUI の「学習ログを取得」で `colab_data/` に取り込めます。
